# 09 · Tablas de la tesis

Reúne, con el formato de la tesis, todas las tablas del documento a partir de las salidas de los
notebooks anteriores. Cada tabla queda en `resultados/tablas/tesis/` con el nombre de su
etiqueta en LaTeX.

| Archivo | Tabla de la tesis | Fuente |
|---|---|---|
| `tab_asimetria.csv` | Asimetría educativa en la caída de la TGF, 2016 y 2024 | notebook 05 |
| `tab_tempo_mac.csv` | Descomposición de la edad media a la maternidad | notebook 05 |
| `tab_tempo_estrato.csv` | Variación de la MAC truncada por nivel educativo | notebook 05 |
| `tab_contribucion_edad.csv` | Contribución de cada tramo de edad a la caída de la TGF | notebook 05 |
| `tab_comparacion_modelos.csv` | Comparación de especificaciones para la cantidad total de hijos | notebook 04 |
| `tab_composicion_base.csv` | Población base 2025 por tercil de ingreso por adulto y grupo etario | notebook 04 |
| `tab_generaciones.csv` | *Completed fertility* por generación, Escenario Base y Alternativo | notebook 07 |
| `tab_validacion_nacional.csv` | TBN de la EPH y de las estadísticas vitales, total del país | notebook 05 |
| `tab_validacion_externa.csv` | TBN de la EPH y de las estadísticas vitales, por grupo de jurisdicciones | notebook 05 |
| `tab_factor_edad0.csv` | Factores anuales de ajuste de los nacimientos recientes | notebook 04 |
| `tab_clima_cinco.csv` | Mujeres de 15 a 49 años por clima educativo del hogar | notebook 05 |
| `tab_heterogeneidad_drift.csv` | Heterogeneidad del *drift* entre estratos, por vector | notebooks 03 y 04 |
| `tab_validacion_vectores.csv` | Métricas de validación de los modelos de clasificación | notebooks 03 y 04 |
| `tab_calibracion_fertilidad.csv` | Calibración del modelo de fertilidad | notebook 04 |

**Tiempo aproximado:** unos segundos.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
DESC = Path("resultados/tablas/descriptivas")
RES = Path("resultados/tablas/resultados")
DIAG = Path("resultados/intermedios/diagnosticos")
SALIDA = Path("resultados/tablas/tesis")
SALIDA.mkdir(parents=True, exist_ok=True)


def guardar(df, nombre):
    df.to_csv(SALIDA / f"{nombre}.csv", index=False)
    return df

## Estadística descriptiva

In [2]:
t1 = pd.read_csv(DESC / "tabla1_asimetria_educativa.csv")
t0 = pd.read_csv(DESC / "tabla0_composicion_mujeres_edu3.csv").set_index("edu3")["share_pct"]
guardar(pd.DataFrame({
    "nivel_educativo": [f"{n} ({t0[n]:.1f}%)" if n in t0 else n for n in t1["nivel_educativo"]],
    "TGF_2016": t1["TGF_2016"].round(2), "TGF_2024": t1["TGF_2024"].round(2),
    "variacion_relativa_pct": (100 * t1["variacion_relativa"]).round(1),
    "significativa_95": np.where(t1["p_valor_bootstrap"] < 0.05, "*", "")}), "tab_asimetria")

,nivel_educativo,TGF_2016,TGF_2024,variacion_relativa_pct,significativa_95
0,Bajo (12.0%),2.42,2.40,-0.9,
1,Medio (50.4%),1.97,1.45,-26.2,*
2,Alto (37.6%),1.54,0.82,-46.8,*
3,Total,1.84,1.20,-34.7,*


In [3]:
t2 = pd.read_csv(DESC / "tabla2_espejismo_tempo.csv").set_index("metrica")
mac = t2.loc[["MAC original (15-49)", "MAC truncada (20-49)"]]
guardar(pd.DataFrame({"": ["2016", "2024", "Diferencia (años)", "Valor p (bootstrap)"],
                      "MAC original (15-49)": [round(v, 3 if i == 3 else 2) for i, v in enumerate(
                          mac.loc["MAC original (15-49)", ["valor_2016", "valor_2024", "diferencia_anios",
                                                           "p_valor_bootstrap"]])],
                      "MAC truncada (20-49)": [round(v, 3 if i == 3 else 2) for i, v in enumerate(
                          mac.loc["MAC truncada (20-49)", ["valor_2016", "valor_2024", "diferencia_anios",
                                                           "p_valor_bootstrap"]])]}), "tab_tempo_mac")
print(f"Composición vs. postergación en la MAC: "
      f"{100 * t2.loc['MAC original (15-49)', 'share_efecto_composicion']:.1f}% / "
      f"{100 * t2.loc['MAC original (15-49)', 'share_postergacion_real']:.1f}%; "
      f"primera maternidad: {100 * t2.loc['Edad 1.ª maternidad original (15-49)', 'share_efecto_composicion']:.1f}% / "
      f"{100 * t2.loc['Edad 1.ª maternidad original (15-49)', 'share_postergacion_real']:.1f}%")

t3 = pd.read_csv(DESC / "tabla3_tempo_por_estrato.csv")
t3 = t3[t3["nivel_educativo"] != "Total"]
guardar(pd.DataFrame({"nivel_educativo": t3["nivel_educativo"], "variacion_MAC_anios": t3["diferencia_anios"].round(2),
                      "significativa_95": np.where(t3["p_valor_bootstrap"] < 0.05, "Sí", "No"),
                      "p_valor": t3["p_valor_bootstrap"].round(3)}), "tab_tempo_estrato")

t4 = pd.read_csv(DESC / "tabla4_contribucion_edad.csv")
tab4 = pd.DataFrame({"tramo_edad": t4["tramo_edad"], "tasa_2016": t4["ASFR_2016_x1000"].round(1),
                     "tasa_2024": t4["ASFR_2024_x1000"].round(1),
                     "variacion_pct": (100 * t4["variacion_relativa_ASFR"]).round(1),
                     "contribucion_a_la_caida_pct": t4["contribucion_a_la_caida_TGF_%"].round(1)})
total = t1[t1["nivel_educativo"] == "Total"].iloc[0]
tab4.loc[len(tab4)] = ["Total (TGF)", round(total["TGF_2016"], 2), round(total["TGF_2024"], 2),
                       round(100 * total["variacion_relativa"], 1), 100.0]
guardar(tab4, "tab_contribucion_edad")

Composición vs. postergación en la MAC: 71.6% / 28.4%; primera maternidad: 68.9% / 31.1%


,tramo_edad,tasa_2016,tasa_2024,variacion_pct,contribucion_a_la_caida_pct
0,15-19,34.20,10.5,-69.2,18.8
1,20-24,53.80,50.7,-5.9,3.8
2,25-29,91.70,60.4,-34.1,23.6
3,30-34,104.90,46.2,-55.9,47.0
4,35-39,61.80,58.1,-6.0,2.3
5,40-44,20.10,13.3,-34.1,5.2
6,45-49,2.70,3.8,42.2,-0.8
7,Total (TGF),1.84,1.2,-34.7,100.0


## Metodología

In [4]:
mh = pd.read_csv(DIAG / "metricas_hijos.csv")
nombres = {"MCO": "MCO", "Lasso": "Lasso", "Ridge": "Ridge", "RF": "Random Forest", "XGB": "XGBoost"}
guardar(pd.DataFrame({"especificacion": mh["Modelo"].map(nombres), "R2": mh["R2"].round(4),
                      "RMSE": mh["RMSE"].round(3), "MAE": mh["MAE"].round(3)}), "tab_comparacion_modelos")

pob = pd.read_csv("resultados/intermedios/poblacion_base/poblacion_base.csv")
pob["grupo_etario"] = pd.cut(pob["age"], [-1, 14, 64, 200], labels=["0-14 años", "15-64 años", "65 años y más"])
comp = pob.pivot_table(index="grupo_etario", columns="status", values="n", aggfunc="sum", observed=True)
comp = (comp / comp.values.sum() * 100).round(1)
comp.columns = ["Tercil bajo", "Tercil medio", "Tercil alto"]
guardar(comp.reset_index(), "tab_composicion_base")

,grupo_etario,Tercil bajo,Tercil medio,Tercil alto
0,0-14 años,7.7,7.8,7.6
1,15-64 años,21.6,20.9,21.9
2,65 años y más,4.3,4.6,3.6


## Resultados

In [5]:
gen = pd.read_csv(RES / "fecundidad_generacional_base.csv")[["generacion", "descendencia_final"]].merge(
    pd.read_csv(RES / "fecundidad_generacional_rebote.csv")[["generacion", "descendencia_final"]],
    on="generacion", suffixes=("_base", "_alternativo"))
guardar(gen.round(2).rename(columns={"descendencia_final_base": "hijos_promedio_base",
                                     "descendencia_final_alternativo": "hijos_promedio_alternativo"}),
        "tab_generaciones")

,generacion,hijos_promedio_base,hijos_promedio_alternativo
0,Generación X (1965-1980),1.90,1.90
1,Millennials (1981-1996),1.85,1.85
2,Centennials (1997-2012),1.15,1.23
3,Generación Alfa (2013-2025),1.12,1.25
4,Generación Beta (2026-2039),0.99,1.21


## Apéndice

In [6]:
vn = pd.read_csv(DESC / "validacion_nacional.csv")
vn = pd.concat([vn, pd.DataFrame([{"year": "Promedio", "cociente_original": vn["cociente_original"].mean(),
                                   "cociente_ajustado": vn["cociente_ajustado"].mean()}])], ignore_index=True)
guardar(pd.DataFrame({"anio": vn["year"], "tbn_eph_original": vn["tbn_eph_original"].round(1),
                      "tbn_estadisticas_vitales": vn["tbn_estadisticas_vitales"].round(1),
                      "cociente_eph_original": vn["cociente_original"].round(2),
                      "cociente_eph_ajustada": vn["cociente_ajustado"].round(2)}), "tab_validacion_nacional")

vj = pd.read_csv(DESC / "validacion_jurisdicciones.csv").set_index("grupo")
guardar(pd.DataFrame({"indicador": ["Cobertura de la EPH", "Casos de edad 0 en la EPH", "Cociente, EPH original",
                                    "Cociente, EPH ajustada", "IC95 original", "IC95 ajustada"],
                      **{g: [round(vj.loc[g, "cobertura_eph"], 2), int(vj.loc[g, "casos_edad0"]),
                             round(vj.loc[g, "cociente_original"], 2), round(vj.loc[g, "cociente_ajustado"], 2),
                             vj.loc[g, "IC95_original"], vj.loc[g, "IC95_ajustado"]] for g in vj.index}}),
       "tab_validacion_externa")

fe = pd.read_csv("resultados/intermedios/eph_corregida/factores_correccion_edad0.csv").dropna(subset=["factor_anual"])
fe = pd.DataFrame({"anio": fe["year"].astype(int).astype(str), "r_T": fe["ratio_bruto_r"].round(3),
                   "g_T": fe["crecimiento_generico_g"].round(3), "f_T": fe["factor_anual"].round(3)})
fe.loc[len(fe)] = ["Mediana", np.nan, np.nan, round(pd.to_numeric(fe["f_T"]).median(), 3)]
guardar(fe, "tab_factor_edad0")

cc = pd.read_csv(DESC / "clima_educativo_cinco_categorias.csv")
guardar(cc.round(1), "tab_clima_cinco")

,clima_educativo_hogar,2016,2024
0,Muy bajo (menos de 7 años),14.2,7.0
1,Bajo (7 a menos de 11),39.9,34.7
2,Medio (11 a menos de 14),30.2,37.7
3,Alto (14 a menos de 16),11.8,14.8
4,Muy alto (16 o más),3.9,5.7


In [7]:
def heterogeneidad(pend, var_pend, peso):
    """Q de Cochran, su p-valor y el peso propio medio (%) sobre los estratos con pendiente propia."""
    s, v, w = map(lambda x: np.asarray(x, dtype=float), (pend, var_pend, peso))
    ok = np.isfinite(s) & np.isfinite(v) & (v > 0)
    pw = 1 / v[ok]
    Q = float((pw * (s[ok] - (pw * s[ok]).sum() / pw.sum()) ** 2).sum())
    return Q, float(1 - stats.chi2.cdf(Q, ok.sum() - 1)), float(100 * w[ok].mean())


df_fert = pd.read_csv(DIAG / "drift_fertilidad.csv")
dv = pd.read_csv(DIAG / "drift_vectores.csv")
filas = [("Fertilidad", "Hijo en el último año",
          *heterogeneidad(df_fert["beta_group"], df_fert["se_group"] ** 2, df_fert["peso_propio"]))]
for vector, evento, clave in [("Empleo", "Ocupada", "employment_vector (target_work)"),
                              ("Emparejamiento", "En pareja", "partner_vector (target_couple)"),
                              ("Emparejamiento", "Pareja ocupada", "partner_vector (target_swork)"),
                              ("Educación", "Permanencia en el sistema", "education_flow_vector (target_inschool)")]:
    d = dv[dv["vector"] == clave]
    filas.append((vector, evento, *heterogeneidad(d["slope"], d["var_slope"], d["peso_propio"])))
het = pd.DataFrame(filas, columns=["vector", "evento", "Q", "p_valor", "peso_propio_pct"]).round(
    {"Q": 2, "p_valor": 3, "peso_propio_pct": 0})
for vector, evento, prefijo, con_q in [("Emparejamiento", "Clima educativo", "partner_vector (target_cedu", True),
                                       ("Educación", "Nivel máximo", "education_stock_vector (education", False)]:
    res = [heterogeneidad(d["slope"], d["var_slope"], d["peso_propio"])
           for _, d in dv[dv["vector"].str.startswith(prefijo)].groupby("vector")]
    q, _, w = zip(*res)
    het.loc[len(het)] = [vector, evento, f"{min(q):.0f} a {max(q):.0f}" if con_q else "--", "--",
                         f"{min(w):.0f} a {max(w):.0f}"]
guardar(het, "tab_heterogeneidad_drift")

,vector,evento,Q,p_valor,peso_propio_pct
0,Fertilidad,Hijo en el último año,16.24,0.039,47.0
1,Empleo,Ocupada,7.55,0.479,0.0
2,Emparejamiento,En pareja,21.73,0.005,57.0
3,Emparejamiento,Pareja ocupada,17.01,0.017,53.0
4,Educación,Permanencia en el sistema,112.84,0.0,68.0
5,Emparejamiento,Clima educativo,8 a 61,--,15 a 82
6,Educación,Nivel máximo,--,--,63 a 99


In [8]:
mm = pd.read_csv(DIAG / "metricas_maternidad.csv").iloc[0]
mv = pd.read_csv(DIAG / "metricas_vectores.csv").set_index("target")
filas = [("Fertilidad", "Hijo en el último año", mm["Tasa_base_test"], mm["AUC_ROC"], mm["AUC_PR"], mm["Brier"],
          mm["Log_Loss"])]
for vector, evento, target in [("Empleo", "Ocupada", "target_work"), ("Emparejamiento", "En pareja", "target_couple"),
                               ("Emparejamiento", "Pareja ocupada", "target_swork"),
                               ("Emparejamiento", "Clima educativo (5 clases)", "target_cedu"),
                               ("Educación", "Permanencia en el sistema", "target_inschool"),
                               ("Educación", "Nivel máximo (7 clases)", "education")]:
    r = mv.loc[target]
    filas.append((vector, evento, r["tasa_base"], r["roc_auc"], r["pr_auc"], r["brier"], r["log_loss"]))
guardar(pd.DataFrame(filas, columns=["vector", "evento", "tasa_base", "ROC", "PR", "Brier", "log_loss"]).round(3),
        "tab_validacion_vectores")

cal = pd.read_csv(DIAG / "calibracion_maternidad.csv").reset_index(drop=True)
guardar(pd.DataFrame({"decil": range(1, len(cal) + 1),
                      "probabilidad_predicha_pct": (100 * cal["prob_predicha_media"]).round(2),
                      "frecuencia_observada_pct": (100 * cal["frac_positivos_empirica"]).round(2)}),
        "tab_calibracion_fertilidad")

,decil,probabilidad_predicha_pct,frecuencia_observada_pct
0,1,0.06,0.25
1,2,0.35,0.35
2,3,0.51,0.96
3,4,0.92,1.03
4,5,1.66,1.52
5,6,2.43,2.41
6,7,3.52,2.71
7,8,5.68,5.54
8,9,8.67,8.46
9,10,14.08,15.07


## Vista rápida de todas las tablas

In [9]:
for f in sorted(SALIDA.glob("*.csv")):
    print(f"\n=== {f.stem} ===")
    print(pd.read_csv(f).to_string(index=False))


=== tab_asimetria ===
nivel_educativo  TGF_2016  TGF_2024  variacion_relativa_pct significativa_95
   Bajo (12.0%)      2.42      2.40                    -0.9              NaN
  Medio (50.4%)      1.97      1.45                   -26.2                *
   Alto (37.6%)      1.54      0.82                   -46.8                *
          Total      1.84      1.20                   -34.7                *

=== tab_calibracion_fertilidad ===
 decil  probabilidad_predicha_pct  frecuencia_observada_pct
     1                       0.06                      0.25
     2                       0.35                      0.35
     3                       0.51                      0.96
     4                       0.92                      1.03
     5                       1.66                      1.52
     6                       2.43                      2.41
     7                       3.52                      2.71
     8                       5.68                      5.54
     9          